# 11 · RawKernel & 커널 개념 적용 기술

> **CuPy 2일 집중 코스 — Day 2 / 단원 8 (커널 심화)**

`RawKernel`은 **CUDA C 소스 전체**를 직접 작성하고 grid/block을 직접 지정하는 가장 낮은 수준의 방법입니다.
여기서 07의 커널 개념(인덱싱·coalescing·공유메모리·occupancy)을 **CUDA C로 직접 구현**하고,
커널을 빠르게 만드는 **개념 적용(최적화) 기술**을 정리합니다.

## 이 노트북에서 구현하는 개념 (07 참조)
- **2 인덱싱**(직접) · **5 coalescing** · **4 공유메모리(타일링/리덕션)** · **7 occupancy**(블록 튜닝)

## 학습 목표
- `RawKernel`로 1D·2D 커널을 작성하고 grid/block을 직접 지정한다.
- 공유메모리·`__syncthreads`로 **블록 리덕션**을 구현한다.
- 커널 최적화 기술(coalescing·타일링·분기 최소화·튜닝)을 적용한다.

> 본 과정 차별성('CUDA C 없이 Python만')상 RawKernel은 **심화/참고**입니다 — 같은 개념을 08~09는 Numba(Python)로 구현했습니다.

## 목차
1. [RawKernel 기초 (saxpy)](#1)
2. [2D 스텐실](#2)
3. [블록 크기 튜닝 (occupancy)](#3)
4. [커널 개념 적용(최적화) 기술](#4)
5. [공유메모리 블록 리덕션](#5)
6. [연습](#6)
7. [체크포인트](#7)

In [ ]:
import os, sys, time, math
import numpy as np
import cupy as cp
from course_utils import print_env, bench, gpu_ms, allclose
print_env()

<a id="1"></a>
## 1. RawKernel 기초 (saxpy)

CUDA C로 `y = a*x + b` 를 작성합니다. 전역 인덱스(07의 2절)를 손으로 계산하고 경계를 검사합니다.
런치: `kernel((blocks,), (threads,), (args...))`.

In [ ]:
saxpy_src = r'''
extern "C" __global__
void saxpy(const float* x, float* y, float a, float b, int n){
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) y[i] = a * x[i] + b;
}'''
saxpy = cp.RawKernel(saxpy_src, 'saxpy')
n = 1 << 20
x = cp.random.rand(n, dtype=cp.float32); y = cp.empty_like(x)
threads = 256; blocks = (n + threads - 1) // threads
saxpy((blocks,), (threads,), (x, y, cp.float32(2.0), cp.float32(1.0), np.int32(n)))
cp.cuda.Device().synchronize()
allclose(2.0*cp.asnumpy(x)+1.0, y, rtol=1e-5, atol=1e-5, name='saxpy')

<a id="2"></a>
## 2. 2D 스텐실

2D 인덱싱(`blockIdx/threadIdx`의 x·y)과 경계 처리를 직접 다룹니다. grid/block을 2D 튜플로 지정.

In [ ]:
stencil_src = r'''
extern "C" __global__
void stencil5(const float* x, float* y, int nx, int ny){
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    int j = blockIdx.y * blockDim.y + threadIdx.y;
    if (i <= 0 || j <= 0 || i >= nx-1 || j >= ny-1) return;
    int idx = j * nx + i;
    y[idx] = 0.25f*(x[idx-nx] + x[idx+nx] + x[idx-1] + x[idx+1]) - x[idx];
}'''
stencil5 = cp.RawKernel(stencil_src, 'stencil5')
nx, ny = 2048, 2048
x_np = np.random.rand(ny, nx).astype(np.float32)
def stencil_np(x):
    y = np.zeros_like(x)
    y[1:-1,1:-1] = 0.25*(x[:-2,1:-1]+x[2:,1:-1]+x[1:-1,:-2]+x[1:-1,2:]) - x[1:-1,1:-1]
    return y
ref = stencil_np(x_np)
xg = cp.asarray(x_np).ravel(); yg = cp.zeros_like(xg)
block = (16, 16); grid = (math.ceil(nx/16), math.ceil(ny/16))
stencil5(grid, block, (xg, yg, np.int32(nx), np.int32(ny)))
cp.cuda.Device().synchronize()
allclose(ref, yg.reshape(ny, nx), rtol=1e-5, atol=1e-5, name='stencil5')

<a id="3"></a>
## 3. 블록 크기 튜닝 (occupancy)

블록 모양에 따라 occupancy·메모리 효율이 달라져 성능이 변합니다(07의 7절). 직접 스윕해 최적점을 찾습니다.

In [ ]:
for block in [(8,8),(16,16),(32,8),(32,16)]:
    grid = (math.ceil(nx/block[0]), math.ceil(ny/block[1]))
    def run(g=grid, b=block): stencil5(g, b, (xg, yg, np.int32(nx), np.int32(ny)))
    print('block', block, '->', round(gpu_ms(bench(run, n_repeat=20, n_warmup=5)), 4), 'ms')

<a id="4"></a>
## 4. 커널 개념 적용(최적화) 기술

07의 개념을 실제 커널에 적용하는 대표 기술입니다.

| 기술 | 적용 개념 | 효과 |
|------|-----------|------|
| **연속 접근**(grid-stride) | 5 coalescing | 메모리 트랜잭션↓·대역폭↑ |
| **공유메모리 타일링/리덕션** | 4 메모리계층 | 전역 접근↓·재사용↑ |
| **분기 최소화** | 3 SIMT | warp divergence↓ |
| **`const`/`__restrict__`** | — | 별칭 없음 가정 → 컴파일러 최적화 |
| **블록/스레드 튜닝** | 7 occupancy | 지연 숨김 |
| **연산 융합** | — | 중간배열·커널 런치↓ |

예: grid-stride 루프로 **연속 접근 + 임의 크기 처리**를 동시에 얻습니다.

In [ ]:
# grid-stride saxpy: warp 인접 스레드가 인접 주소 접근(coalesced), 큰 n도 처리
saxpy_gs_src = r'''
extern "C" __global__
void saxpy_gs(const float* __restrict__ x, float* __restrict__ y, float a, int n){
    int i = blockIdx.x*blockDim.x + threadIdx.x;
    int stride = gridDim.x * blockDim.x;
    for (; i < n; i += stride) y[i] = a*x[i];
}'''
saxpy_gs = cp.RawKernel(saxpy_gs_src, 'saxpy_gs')
n = 1 << 24; x = cp.random.rand(n, dtype=cp.float32); y = cp.empty_like(x)
saxpy_gs((1024,), (256,), (x, y, cp.float32(3.0), np.int32(n)))
cp.cuda.Device().synchronize()
allclose(3.0*cp.asnumpy(x), y, rtol=1e-5, atol=1e-5, name='saxpy_gs')

<a id="5"></a>
## 5. 공유메모리 블록 리덕션 (기술 종합)

합(reduction)을 **공유메모리 + `__syncthreads` + 트리 리덕션**으로 구현합니다 — 07의 개념 4·6을 RawKernel로 직접 적용.
각 블록이 부분합을 만들고, 블록 부분합을 마지막에 합칩니다. 공유메모리 크기는 런치 시 `shared_mem`으로 지정.

In [ ]:
reduce_src = r'''
extern "C" __global__
void block_sum(const float* __restrict__ x, float* partial, int n){
    extern __shared__ float sdata[];
    int tid = threadIdx.x;
    int i = blockIdx.x*blockDim.x + threadIdx.x;
    sdata[tid] = (i < n) ? x[i] : 0.0f;   // coalesced 로드
    __syncthreads();
    for (int s = blockDim.x/2; s > 0; s >>= 1){   // 트리 리덕션
        if (tid < s) sdata[tid] += sdata[tid + s];
        __syncthreads();
    }
    if (tid == 0) partial[blockIdx.x] = sdata[0];
}'''
block_sum = cp.RawKernel(reduce_src, 'block_sum')
n = 1 << 22; x = cp.random.rand(n, dtype=cp.float32)
threads = 256; blocks = (n + threads - 1)//threads
partial = cp.empty(blocks, dtype=cp.float32)
block_sum((blocks,), (threads,), (x, partial, np.int32(n)), shared_mem=threads*4)
total = float(partial.sum())   # 블록 부분합 최종 합산
allclose(float(x.sum()), total, rtol=1e-3, atol=1e-1, name='block_sum')

<a id="6"></a>
## 6. 연습 — clamp RawKernel

`y = min(max(x, lo), hi)` 를 RawKernel(CUDA C)로 작성하세요(1D, 경계 검사·grid-stride 권장).

In [ ]:
# TODO: clamp_src = r''' extern "C" __global__ void clamp(...){ ... } '''
x_np = np.random.randn(1_000_000).astype(np.float32)
ref = np.clip(x_np, -1.0, 1.0)
# clamp = cp.RawKernel(clamp_src, 'clamp'); 런치 후 allclose(ref, ...)

<details><summary>💡 해답 보기</summary>

```python
clamp_src = r'''
extern "C" __global__
void clamp(const float* __restrict__ x, float* __restrict__ y, float lo, float hi, int n){
    int i = blockIdx.x*blockDim.x + threadIdx.x;
    int stride = gridDim.x*blockDim.x;
    for (; i < n; i += stride){ float v = x[i]; v = v<lo?lo:(v>hi?hi:v); y[i] = v; }
}'''
clamp = cp.RawKernel(clamp_src, 'clamp')
xg = cp.asarray(x_np); yg = cp.empty_like(xg); n = xg.size
clamp((1024,), (256,), (xg, yg, cp.float32(-1), cp.float32(1), np.int32(n)))
allclose(ref, yg, rtol=1e-5, atol=1e-5, name='clamp')
```
</details>

<a id="tiled"></a>
## (심화) 공유메모리 타일 transpose

전치(transpose)는 **쓰기가 비연속**이라 느립니다(08 naive). **공유메모리 타일**로 읽기·쓰기를 모두 연속(coalesced)으로 만듭니다:
타일을 공유메모리에 연속으로 읽어 들이고, `__syncthreads` 후 전치해서 연속으로 씁니다. (coalescing + 공유메모리 종합)
뱅크 충돌 회피를 위해 타일 폭을 `TILE+1`로 패딩합니다.

In [ ]:
transpose_src = r'''
#define TILE 32
extern "C" __global__
void transpose_tiled(const float* a, float* out, int n){
    __shared__ float tile[TILE][TILE+1];   // +1: 뱅크 충돌 회피
    int x = blockIdx.x*TILE + threadIdx.x;
    int y = blockIdx.y*TILE + threadIdx.y;
    if (x < n && y < n) tile[threadIdx.y][threadIdx.x] = a[y*n + x];  // 연속 읽기
    __syncthreads();
    int tx = blockIdx.y*TILE + threadIdx.x;
    int ty = blockIdx.x*TILE + threadIdx.y;
    if (tx < n && ty < n) out[ty*n + tx] = tile[threadIdx.x][threadIdx.y];  // 연속 쓰기
}'''
transpose_tiled = cp.RawKernel(transpose_src, 'transpose_tiled')
n = 2048; A = cp.random.rand(n, n, dtype=cp.float32); T = cp.empty_like(A)
block = (32, 32); grid = (n//32, n//32)
transpose_tiled(grid, block, (A.ravel(), T.ravel(), np.int32(n)))
cp.cuda.Device().synchronize()
allclose(cp.asnumpy(A).T, T.reshape(n,n), rtol=1e-5, atol=1e-5, name='tiled transpose')

**연습 — naive와 비교**: 08의 naive transpose(있으면)나 단순 버전과 타일 버전의 속도를 비교하세요(coalescing 효과).

In [ ]:
# 단순(비연속 쓰기) 비교용 RawKernel
naive_src = r'''
extern "C" __global__
void transpose_naive(const float* a, float* out, int n){
    int x = blockIdx.x*blockDim.x + threadIdx.x;
    int y = blockIdx.y*blockDim.y + threadIdx.y;
    if (x<n && y<n) out[x*n + y] = a[y*n + x];   // 쓰기 비연속
}'''
transpose_naive = cp.RawKernel(naive_src, 'transpose_naive')
def run_naive(): transpose_naive(grid, block, (A.ravel(), T.ravel(), np.int32(n)))
def run_tiled(): transpose_tiled(grid, block, (A.ravel(), T.ravel(), np.int32(n)))
print('naive', round(gpu_ms(bench(run_naive)),4), 'ms | tiled', round(gpu_ms(bench(run_tiled)),4), 'ms')

<a id="7"></a>
## 7. 체크포인트

- [ ] RawKernel로 1D(saxpy)·2D(스텐실) 커널을 작성·런치했다
- [ ] 블록 크기 튜닝으로 occupancy 영향을 봤다
- [ ] 최적화 기술(coalescing·타일링·분기·튜닝)을 안다
- [ ] 공유메모리+`__syncthreads`로 블록 리덕션을 구현했다

다음: **`12_interop_frameworks`** — DLPack으로 PyTorch 등과 무복사 연동합니다.